# 📌 Importação de bibliotecas

In [1]:
import numpy as np
import pandas as pd

In [2]:
from typing import Union

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [5]:
from scipy.stats import f_oneway
from scipy.stats import levene
from scipy import stats
import pingouin as pg

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, KFold
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from yellowbrick.model_selection import FeatureImportances

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error, mean_absolute_percentage_error
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import average_precision_score
from sklearn.metrics import classification_report
from yellowbrick.classifier import ClassificationReport
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc, precision_recall_curve
)

In [7]:
import copy

In [8]:
import warnings

## Importação de pacotes locais

In [9]:
import src.local_tools as lt
import src.telecomx_analysis as ta
import src.telecomx_machine_learning as tml

## Configurações do ambiente

In [10]:
pd.set_option('display.max_columns', None)

In [11]:
warnings.simplefilter(action='ignore', category=FutureWarning)

## Constantes

In [12]:
NUM_SEMENTE_ALEATORIA = 42
TAMANHO_TESTE = 0.3
MAXIMO_ITERACAO = 1000

In [13]:
LIST_SCORING = ['accuracy','recall', 'precision', 'f1']

# 📌 Extração de dados

In [14]:
df_dados = pd.read_csv('./dados/dados_tratados.csv')

In [15]:
df_dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customerID                    7043 non-null   object 
 1   Churn                         7043 non-null   int64  
 2   customer_gender               7043 non-null   object 
 3   customer_SeniorCitizen        7043 non-null   int64  
 4   customer_Partner              7043 non-null   int64  
 5   customer_Dependents           7043 non-null   int64  
 6   customer_tenure               7043 non-null   int64  
 7   phone_PhoneService            7043 non-null   int64  
 8   phone_MultipleLines           7043 non-null   int64  
 9   internet_InternetService      7043 non-null   int64  
 10  internet_OnlineSecurity       7043 non-null   int64  
 11  internet_OnlineBackup         7043 non-null   int64  
 12  internet_DeviceProtection     7043 non-null   int64  
 13  int

In [16]:
df_dados.nunique()

customerID                      7043
Churn                              2
customer_gender                    2
customer_SeniorCitizen             2
customer_Partner                   2
customer_Dependents                2
customer_tenure                   72
phone_PhoneService                 2
phone_MultipleLines                2
internet_InternetService           2
internet_OnlineSecurity            2
internet_OnlineBackup              2
internet_DeviceProtection          2
internet_TechSupport               2
internet_StreamingTV               2
internet_StreamingMovies           2
account_Contract                   3
account_PaperlessBilling           2
account_PaymentMethod              4
account_Charges_Monthly         1585
account_Charges_Total           6534
internet_Service_Description       3
customer_tenure_bins               6
account_Charges_Monthly_bins       6
account_Charges_Total_bins        13
account_Contract_Monthly           2
additional_InternetService         7
o

Verificar dados duplicados

In [17]:
df_dados.duplicated().sum()

0

# 📌 Tratamento de dados

In [18]:
df_ohe = tml.df_final_modelo_v1(df_dados)

# 📌 Seleção e validação dos modelos de treinamento

## Tabela de verificação de padronização dos dados

| Modelo                 | Precisa padronizar? | Tipo de padronização (se necessário)   |
| ---------------------- | ------------------- | -------------------------------------- |
| DecisionTreeClassifier | ❌ Não               | —                                      |
| LogisticRegression     | ✅ Sim               | `StandardScaler` (média 0, desvio 1)   |
| RandomForest           | ❌ Não               | —                                      |
| XGBoost                | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| LightGBM               | ⚠️ Opcional         | Pode ajudar (StandardScaler ou MinMax) |
| CatBoost               | ❌ Não               | —                                      |


## Conceitos sobre as métricas de um modelo

Aqui está o quadro comparativo para churn (evasão de clientes), considerando que a classe positiva é “cliente vai sair”:

| **Métrica**                | **O que mede**                                                                     | **Quando valor é alto**                                        | **Risco quando valor é baixo**                                   | **Custo de erro associado**                                                |
| -------------------------- | ---------------------------------------------------------------------------------- | -------------------------------------------------------------- | ---------------------------------------------------------------- | -------------------------------------------------------------------------- |
| **Precisão (Precision)**   | Entre todos que o modelo previu como “vai sair”, qual porcentagem realmente saiu   | Você gasta retenção apenas em quem de fato sairia              | Gastar recursos em retenção de clientes que iam ficar (FP alto)  | **Custo financeiro** com campanhas desnecessárias (descontos, bônus, etc.) |
| **Recall (Sensibilidade)** | Entre todos que realmente saíram, qual porcentagem o modelo previu como “vai sair” | Você identifica a maior parte dos clientes que iam sair        | Deixar escapar clientes que saem (FN alto)                       | **Perda de receita** e potencial perda de market share                     |
| **F1-Score**               | Média harmônica de precisão e recall                                               | Equilíbrio entre acertar quem vai sair e evitar falsos alarmes | Ou alto custo de retenção inútil ou perda de clientes — ou ambos | **Equilíbrio financeiro e estratégico** — custo total menor                |
| **FP (Falso Positivo)**    | Previu saída, mas o cliente ficaria                                                | —                                                              | Gastar para reter quem não ia sair                               | Desperdício de budget de retenção                                          |
| **FN (Falso Negativo)**    | Previu permanência, mas o cliente saiu                                             | —                                                              | Não agir para reter quem realmente sairia                        | Perda direta de receita + possível impacto na reputação                    |
    

📌 Resumo visual da prioridade

    Se retenção for muito cara → priorizar alta precisão.

    Se perder clientes for muito prejudicial → priorizar alto recall.

    Se quer balancear ambos → otimizar F1-Score.

## Split base de dados em train e test

In [19]:
X = df_ohe.drop(columns=['Churn'])

In [20]:
y = df_ohe.Churn

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = NUM_SEMENTE_ALEATORIA, test_size=TAMANHO_TESTE, stratify=y)

## Treinamento do Modelo - CatBoost

In [22]:
from catboost import CatBoostClassifier

In [23]:
model_cbc = CatBoostClassifier(
    n_estimators=300,
    random_state=NUM_SEMENTE_ALEATORIA
)

In [24]:
model_cbc.fit(X_train, y_train)

Learning rate set to 0.061413
0:	learn: 0.6533179	total: 218ms	remaining: 1m 5s
1:	learn: 0.6216230	total: 225ms	remaining: 33.6s
2:	learn: 0.5942212	total: 231ms	remaining: 22.8s
3:	learn: 0.5687756	total: 236ms	remaining: 17.5s
4:	learn: 0.5483945	total: 244ms	remaining: 14.4s
5:	learn: 0.5304279	total: 250ms	remaining: 12.2s
6:	learn: 0.5166546	total: 256ms	remaining: 10.7s
7:	learn: 0.5039794	total: 261ms	remaining: 9.51s
8:	learn: 0.4937180	total: 265ms	remaining: 8.56s
9:	learn: 0.4859991	total: 269ms	remaining: 7.79s
10:	learn: 0.4772230	total: 274ms	remaining: 7.19s
11:	learn: 0.4700168	total: 278ms	remaining: 6.66s
12:	learn: 0.4646940	total: 281ms	remaining: 6.21s
13:	learn: 0.4589160	total: 286ms	remaining: 5.84s
14:	learn: 0.4547355	total: 290ms	remaining: 5.5s
15:	learn: 0.4499822	total: 293ms	remaining: 5.2s
16:	learn: 0.4476263	total: 297ms	remaining: 4.94s
17:	learn: 0.4446294	total: 300ms	remaining: 4.7s
18:	learn: 0.4420637	total: 305ms	remaining: 4.51s
19:	learn: 0.4

In [25]:
# 8️⃣ Fazer previsões
y_pred_cbc = model_cbc.predict(X_test)
y_pred_proba_cbc = model_cbc.predict(X_test)

### Relatório de métricas

In [26]:
tml.avaliar_modelo(y_test, y_pred_cbc, y_pred_proba_cbc, False)

Métricas:
Acurácia: 0.7837
Precisão: 0.6150
Recall: 0.4955
F1-Score: 0.5489
ROC AUC: 0.6917

Relatório de classificação:

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1552
           1       0.62      0.50      0.55       561

    accuracy                           0.78      2113
   macro avg       0.72      0.69      0.70      2113
weighted avg       0.77      0.78      0.78      2113


Matrix de confusão:

[[1378  174]
 [ 283  278]]


In [27]:
tml.df_classifier_metrics(y_test, y_pred_cbc, y_pred_proba_cbc, ['CatBoost'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
CatBoost,0.78372,0.615044,0.495544,0.548865,0.691715,2113,1378,174,283,278


In [28]:
list_df = []
colunas_binarias = lt.identify_columns_binary_values(X_test)
for c in colunas_binarias:
    list_df.append(tml.df_specific_confusion_matrix(X_test, y_test, y_pred_proba_cbc, c))
df_independent = pd.concat(list_df, axis=0)

E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes - Parte 2\TelecomX_parte2_BR\scripts\telecomx_machine_learning.py:251: RuntimeWarning: invalid value encountered in scalar divide
  precision = tp / (tp + fp)
E:\Local_Drive\Cursos Livres\ONE\06-Estatística e Machine Learning G8 - ONE\08-Challenge Telecom X - análise de evasão de clientes - Parte 2\TelecomX_parte2_BR\scripts\telecomx_machine_learning.py:253: RuntimeWarning: invalid value encountered in scalar divide
  f1_score = (2 * precision * recall) / (precision + recall)


In [29]:
df_independent.reset_index()

,index,filter_value,support,accuracy,negative predictive value,precision,recall,f1-score,TN,FP,FN,TP
0,customer_SeniorCitizen,1,349,0.727794,0.776650,0.664474,0.696552,0.680135,153,51,44,101
1,customer_Partner,1,1032,0.814922,0.855395,0.541353,0.356436,0.429851,769,61,130,72
2,customer_Dependents,1,616,0.840909,0.875220,0.425532,0.219780,0.289855,498,27,71,20
3,phone_MultipleLines,1,883,0.779162,0.830247,0.638298,0.576923,0.606061,538,85,110,150
4,account_PaperlessBilling,1,1213,0.752679,0.806258,0.636126,0.601485,0.618321,670,139,161,243
5,account_Contract_Monthly,1,1182,0.670897,0.705479,0.615044,0.563895,0.588360,515,174,215,278
6,additional_InternetService_0,1,676,0.838757,0.871750,0.646465,0.463768,0.540084,503,35,74,64
7,additional_InternetService_1,1,297,0.626263,0.666667,0.585034,0.632353,0.607774,100,61,50,86
8,additional_InternetService_2,1,291,0.701031,0.738693,0.619565,0.522936,0.567164,147,35,52,57
9,additional_InternetService_3,1,318,0.786164,0.810277,0.692308,0.483871,0.569620,205,20,48,45


### Otimizando os hiperparâmetros com o GridSearchCV

In [30]:
param_grid = {
    'depth': [5, 10],               # profundidade da árvore
    'learning_rate': [0.01, 0.1, 0.2],  # taxa de aprendizado
    'iterations': [100, 300],      # número de árvores
    'min_data_in_leaf': [10, 20],   # mínimo de amostras por folha
    'subsample': [0.6, 1.0],       # fração de amostras
    'rsm': [0.6, 1.0],             # fração de features
    'l2_leaf_reg': [3, 7],        # regularização L2
}

In [31]:
%%timeit
model_cbc = CatBoostClassifier(
    random_state=NUM_SEMENTE_ALEATORIA,
    verbose=0
)

25.9 µs ± 1.13 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [34]:
model_cbc = CatBoostClassifier(
    random_state=NUM_SEMENTE_ALEATORIA,
    verbose=0
)

In [35]:
model_grid_cbc = GridSearchCV(
    estimator=model_cbc,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [36]:
model_grid_cbc.fit(X_train, y_train)

Fitting 5 folds for each of 192 candidates, totalling 960 fits


GridSearchCV(cv=5,
             estimator=<catboost.core.CatBoostClassifier object at 0x000002455BF07D50>,
             n_jobs=-1,
             param_grid={'depth': [5, 10], 'iterations': [100, 300],
                         'l2_leaf_reg': [3, 7],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'min_data_in_leaf': [10, 20], 'rsm': [0.6, 1.0],
                         'subsample': [0.6, 1.0]},
             scoring='recall', verbose=1)

In [37]:
model_grid_cbc.best_params_

{'depth': 5,
 'iterations': 100,
 'l2_leaf_reg': 7,
 'learning_rate': 0.2,
 'min_data_in_leaf': 10,
 'rsm': 1.0,
 'subsample': 1.0}

In [38]:
y_pred_grid_cdc = model_grid_cbc.predict(X_test)

In [40]:
y_pred_proba_grid_cdc = model_grid_cbc.predict_proba(X_test)

In [41]:
tml.df_classifier_metrics(y_test, y_pred_grid_cdc, y_pred_proba_grid_cdc, ['CatBoost - Grid'])

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
CatBoost - Grid,0.791765,0.632385,0.515152,0.56778,0.823744,2113,1384,168,272,289


### Consolidação das métricas

In [44]:
df_metricas = []

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_cbc, y_pred_proba_cbc, ['CatBoost']))

df_metricas.append(tml.df_classifier_metrics(y_test, y_pred_grid_cdc, y_pred_proba_grid_cdc, ['CatBoost - Grid']))

df_metricas = pd.concat(df_metricas, axis=0)
df_metricas

,accuracy,precision,recall,f1,roc_auc,support,True Negative,False Positive,False Negative,True Positive
CatBoost,0.783720,0.615044,0.495544,0.548865,0.691715,2113,1378,174,283,278
CatBoost - Grid,0.791765,0.632385,0.515152,0.567780,0.823744,2113,1384,168,272,289


In [38]:
df_metricas.to_clipboard()